In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("/Users/muna/Hana_research/data/db/Hana_Research.db")

conn = sqlite3.connect(DB_PATH)

# ============================================================
# 1. 対象患者のStudy_ID
# ============================================================

study_ids = pd.read_sql_query(
    """
    SELECT Study_ID
    FROM tolvaptan_study
    WHERE Study_ID IS NOT NULL
    """,
    conn
)

print("対象 Study_ID:", len(study_ids))


# ============================================================
# 2. カルテの疾患関連記載を確認
# ============================================================

keywords = [
    "心不全",
    "心不全症",
    "HF",
    "HFrEF",
    "HFpEF",
    "HFmrEF",
    "腎不全",
    "慢性腎不全",
    "急性腎不全",
    "CKD",
    "腎機能",
    "ネフローゼ",
    "肝硬変",
    "癌",
    "がん",
    "悪性腫瘍",
    "サムスカ",
]

print("\n" + "=" * 70)
print("KARTE DISEASE EXAMPLES")
print("=" * 70)

for word in keywords:

    q = """
        SELECT
            Study_ID,
            visit_datetime,
            record_no,
            substr(karte_text, 1, 500) AS karte_text
        FROM karte
        WHERE Study_ID IN (
            SELECT Study_ID
            FROM tolvaptan_study
            WHERE Study_ID IS NOT NULL
        )
        AND karte_text IS NOT NULL
        AND karte_text LIKE ?
        ORDER BY Study_ID, visit_datetime
        LIMIT 20
    """

    rows = pd.read_sql_query(
        q,
        conn,
        params=[f"%{word}%"]
    )

    print(f"\n### {word}")
    print("n shown:", len(rows))

    if not rows.empty:
        print(rows.to_string(index=False))


# ============================================================
# 3. Crの2つのitem_codeを比較
# ============================================================

print("\n" + "=" * 70)
print("CREATININE: ITEM CODE 0069 vs 0074")
print("=" * 70)

cr = pd.read_sql_query(
    """
    SELECT
        item_code,
        item_name,
        COUNT(*) AS n,
        COUNT(value_num) AS numeric_n,
        MIN(sample_date_parsed) AS first_date,
        MAX(sample_date_parsed) AS last_date
    FROM lab_clean
    WHERE item_code IN ('0069', '0074')
    GROUP BY item_code, item_name
    ORDER BY item_code
    """,
    conn
)

print(cr.to_string(index=False))


# ============================================================
# 4. 対象224人についてCrの実例
# ============================================================

print("\n" + "=" * 70)
print("CREATININE EXAMPLES IN TOLVAPTAN COHORT")
print("=" * 70)

cr_examples = pd.read_sql_query(
    """
    SELECT
        l.Patient_ID,
        l.sample_date,
        l.item_code,
        l.item_name,
        l.value_raw,
        l.value_num,
        l.unit
    FROM lab_clean l
    INNER JOIN tolvaptan_study t
        ON CAST(l.Patient_ID AS INTEGER) = t.Patient_ID
    WHERE l.item_code IN ('0069', '0074')
    ORDER BY l.Patient_ID, l.sample_date
    LIMIT 100
    """,
    conn
)

print(cr_examples.to_string(index=False))


conn.close()

print("\n" + "=" * 70)
print("STEP 1B FINISHED")
print("No database tables were modified.")
print("=" * 70)

対象 Study_ID: 224

KARTE DISEASE EXAMPLES

### 心不全
n shown: 20
Study_ID      visit_datetime record_no                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      karte_text
 P000005 2015/10/27(火) 14:53      3023      ■患者さんおよびご家族のお話\r\n熱っぽくて体がだるい。イライラする。\r\n原因は、毎日夜中にピンポンを鳴らされて眠れない。\r\nマグミットとプルゼニドで自己調節して、毎日排便あり。\r\n血圧は120-140/50-80,PR 50-60台くらい。\r\n息切れなし。\r\nマグミットとメリスロンは残薬たくさんあり。\r\n\r\n食事：良好\r\n\r\n嘔気なし\r\n\r\n排便：問題なし\r\n\r\n睡眠：良好\r\n\r\n眼瞼結膜貧血なし\r\n\r\n皮膚ツルゴール正常\r\n\r\n下腿浮腫：中等度\r\n\r\n■身体所見\

In [18]:
import sqlite3
import pandas as pd
import re
from pathlib import Path

# ============================================================
# Step 2
# tolvaptan開始日の前後2か月のカルテから疾患候補を抽出
# ＋患者ごとに4疾患を0/1化
# ============================================================

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"

OUTPUT_DIR = Path(
    "/Users/muna/Hana_research/src/tolvaptan_study/"
)

CANDIDATE_FILE = OUTPUT_DIR / "step2_disease_candidates_pm2m.csv"

PATIENT_FILE = OUTPUT_DIR / "step2_disease_patient_flags.csv"

# 前後2か月
DAYS_BEFORE = 60
DAYS_AFTER = 60


# ============================================================
# 1. tolvaptan_study の対象患者を取得
# ============================================================

conn = sqlite3.connect(DB_PATH)

tolvaptan = pd.read_sql_query(
    """
    SELECT
        Patient_ID,
        Study_ID,
        ind_date
    FROM tolvaptan_study
    WHERE ind_date IS NOT NULL
    """,
    conn
)

tolvaptan["Patient_ID"] = (
    tolvaptan["Patient_ID"]
    .astype(str)
)

tolvaptan["Study_ID"] = (
    tolvaptan["Study_ID"]
    .astype(str)
)

tolvaptan["ind_date"] = pd.to_datetime(
    tolvaptan["ind_date"],
    errors="coerce"
)

print("=" * 70)
print("tolvaptan_study")
print("=" * 70)
print(f"対象患者数: {len(tolvaptan):,}")
print()


# ============================================================
# 2. 対象患者のカルテを取得
# ============================================================

patient_ids = (
    tolvaptan["Patient_ID"]
    .dropna()
    .unique()
    .tolist()
)

placeholders = ",".join(
    ["?"] * len(patient_ids)
)

karte = pd.read_sql_query(
    f"""
    SELECT
        Patient_ID,
        Study_ID,
        visit_datetime,
        karte_text
    FROM karte
    WHERE Patient_ID IN ({placeholders})
      AND karte_text IS NOT NULL
      AND karte_text != ''
    """,
    conn,
    params=patient_ids
)

conn.close()

karte["Patient_ID"] = (
    karte["Patient_ID"]
    .astype(str)
)

print("=" * 70)
print("カルテ")
print("=" * 70)
print(f"取得カルテ件数: {len(karte):,}")
print()


# ============================================================
# 3. カルテ日付を作成
# ============================================================

karte["visit_date"] = pd.to_datetime(
    karte["visit_datetime"]
    .astype(str)
    .str.extract(r"^(\d{4}/\d{1,2}/\d{1,2})", expand=False),
    errors="coerce"
)


# ============================================================
# 4. tolvaptan_study と結合
# ============================================================

tolvaptan_merge = (
    tolvaptan[
        [
            "Patient_ID",
            "Study_ID",
            "ind_date"
        ]
    ]
    .rename(columns={"Study_ID": "Study_ID_tolva"})
)

karte = karte.merge(
    tolvaptan_merge,
    on="Patient_ID",
    how="inner"
)


# ============================================================
# 5. ind_date 前後60日のカルテだけ残す
# ============================================================

karte["days_from_ind"] = (
    karte["visit_date"]
    - karte["ind_date"]
).dt.days

karte_window = karte[
    karte["visit_date"].notna()
    & karte["ind_date"].notna()
    & (karte["days_from_ind"] >= -DAYS_BEFORE)
    & (karte["days_from_ind"] <= DAYS_AFTER)
].copy()

print("=" * 70)
print("ind_date 前後60日のカルテ")
print("=" * 70)
print(f"件数: {len(karte_window):,}")
print()


# ============================================================
# 6. カルテ本文を検索用に正規化
# ============================================================

karte_window["karte_text_search"] = (
    karte_window["karte_text"]
    .astype(str)
    .str.upper()
)


# ============================================================
# 7. 疾患候補の検索パターン
# ============================================================

patterns = {

    "heart_failure": [
        r"心不全",
        r"\bHF\b",
        r"HFPEF",
        r"HFREF",
        r"HFMR EF",
        r"HFMR-EF",
        r"\bCHF\b",
    ],

    "renal_failure": [
        r"腎不全",
        r"腎機能不全",
        r"腎機能低下",
        r"慢性腎臓病",
        r"慢性腎疾患",
        r"\bCKD\b",
        r"末期腎不全",
        r"腎不全末期",
        r"ネフローゼ",
        r"NEPHROTIC",
        r"透析",
        r"人工透析",
        r"血液透析",
        r"腹膜透析",
        r"\bESRD\b",
    ],

    "cirrhosis": [
        r"肝硬変",
        r"非代償性肝硬変",
        r"代償性肝硬変",
        r"アルコール性肝硬変",
        r"CIRRHOSIS",
    ],

    "cancer": [
        r"癌",
        r"がん",
        r"悪性腫瘍",
        r"悪性新生物",
        r"癌腫",
        r"CANCER",
        r"MALIGNANCY",
    ],
}


# ============================================================
# 8. 疾患候補を抽出
# ============================================================

hits = []

for disease, disease_patterns in patterns.items():

    regex = re.compile(
        "|".join(disease_patterns),
        flags=re.IGNORECASE
    )

    mask = (
        karte_window["karte_text_search"]
        .str.contains(
            regex,
            na=False,
            regex=True
        )
    )

    tmp = karte_window.loc[
        mask,
        [
            "Study_ID_tolva",
            "Patient_ID",
            "ind_date",
            "visit_date",
            "days_from_ind",
            "visit_datetime",
            "karte_text",
        ]
    ].copy()

    tmp["disease"] = disease

    hits.append(tmp)


# ============================================================
# 9. 疾患候補を結合
# ============================================================

hits = pd.concat(
    hits,
    ignore_index=True
)

hits = hits.rename(
    columns={
        "Study_ID_tolva": "Study_ID"
    }
)

hits = hits[
    [
        "Study_ID",
        "Patient_ID",
        "ind_date",
        "disease",
        "visit_date",
        "days_from_ind",
        "visit_datetime",
        "karte_text",
    ]
]


# ============================================================
# 10. 疾患候補カルテを保存
# ============================================================

hits.to_csv(
    CANDIDATE_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 11. 224人全員を母集団にして0/1化
# ============================================================

patient_flags = tolvaptan[
    [
        "Study_ID",
        "Patient_ID",
        "ind_date"
    ]
].copy()

# 4疾患をすべて0で初期化
for disease in patterns.keys():
    patient_flags[disease] = 0


# ============================================================
# 12. 疾患ごとに、候補があれば1
# ============================================================

for disease in patterns.keys():

    disease_patients = set(
        hits.loc[
            hits["disease"] == disease,
            "Study_ID"
        ]
    )

    patient_flags.loc[
        patient_flags["Study_ID"].isin(disease_patients),
        disease
    ] = 1


# ============================================================
# 13. 患者ごとの0/1表を保存
# ============================================================

patient_flags.to_csv(
    PATIENT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 14. 結果確認
# ============================================================

print("=" * 70)
print("Step 2 完了")
print("=" * 70)

print(f"疾患候補カルテ:")
print(CANDIDATE_FILE)
print()

print(f"患者別0/1データ:")
print(PATIENT_FILE)
print()


# ------------------------------------------------------------
# 各疾患の患者数
# ------------------------------------------------------------

print("=" * 70)
print("各疾患の患者数")
print("=" * 70)

for disease in patterns.keys():

    n = patient_flags[disease].sum()

    print(
        f"{disease:20s}: {n:3d} / {len(patient_flags):3d}"
    )

print()


# ------------------------------------------------------------
# 4疾患の患者別0/1表
# ------------------------------------------------------------

print("=" * 70)
print("患者別疾患フラグ")
print("=" * 70)

print(
    patient_flags[
        [
            "Study_ID",
            "heart_failure",
            "renal_failure",
            "cirrhosis",
            "cancer",
        ]
    ].head(20).to_string(index=False)
)

print()


# ------------------------------------------------------------
# 各疾患のヒット件数
# ------------------------------------------------------------

print("=" * 70)
print("疾患候補ヒット件数")
print("=" * 70)

print(
    hits["disease"]
    .value_counts()
)

print()

print("完了")

tolvaptan_study
対象患者数: 224

カルテ
取得カルテ件数: 27,896

ind_date 前後60日のカルテ
件数: 6,122

Step 2 完了
疾患候補カルテ:
/Users/muna/Hana_research/src/tolvaptan_study/step2_disease_candidates_pm2m.csv

患者別0/1データ:
/Users/muna/Hana_research/src/tolvaptan_study/step2_disease_patient_flags.csv

各疾患の患者数
heart_failure       : 184 / 224
renal_failure       :  82 / 224
cirrhosis           :  16 / 224
cancer              :  91 / 224

患者別疾患フラグ
Study_ID  heart_failure  renal_failure  cirrhosis  cancer
 P000005              1              0          0       0
 P000010              1              0          0       0
 P000028              1              1          0       1
 P000041              1              1          0       1
 P000061              1              0          1       1
 P000067              1              0          0       0
 P000068              1              0          0       0
 P000090              0              0          0       1
 P000103              1              0          0       0
 P000

In [ ]:
# ============================================================
# Step 3
# 224人全員を母集団として、4疾患を暫定0/1化
# ============================================================

import sqlite3
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# ファイル・DB
# ------------------------------------------------------------

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"

INPUT_FILE = Path(
    "/Users/muna/Hana_research/src/tolvaptan_study/"
    "step2_disease_candidates.csv"
)

OUTPUT_FILE = Path(
    "/Users/muna/Hana_research/src/tolvaptan_study/"
    "step3_disease_flags.csv"
)


# ------------------------------------------------------------
# 1. tolvaptan_studyから「224人全員」を取得
# ------------------------------------------------------------

conn = sqlite3.connect(DB_PATH)

patients = pd.read_sql_query(
    """
    SELECT
        Patient_ID,
        Study_ID,
        ind_date
    FROM tolvaptan_study
    WHERE ind_date IS NOT NULL
    """,
    conn
)

conn.close()


# 型をそろえる
patients["Patient_ID"] = patients["Patient_ID"].astype(str)
patients["Study_ID"] = patients["Study_ID"].astype(str)


print("=" * 70)
print("Step 3：母集団")
print("=" * 70)

print(f"tolvaptan_study対象患者数: {len(patients):,}")
print()


# ------------------------------------------------------------
# 2. Step 2の疾患候補を読み込む
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df["Study_ID"] = df["Study_ID"].astype(str)
df["Patient_ID"] = df["Patient_ID"].astype(str)


print("=" * 70)
print("Step 2候補")
print("=" * 70)

print(f"候補カルテ件数: {len(df):,}")
print(f"候補が存在する患者数: {df['Study_ID'].nunique():,}")
print()


# ------------------------------------------------------------
# 3. 224人全員を母集団にする
# ------------------------------------------------------------

patient_info = patients.copy()


# ------------------------------------------------------------
# 4. 4疾患を0/1化
#
# 候補記載が1件でもあれば 1
# 候補記載がなければ 0
# ------------------------------------------------------------

disease_list = [
    "heart_failure",
    "renal_failure",
    "cirrhosis",
    "cancer",
]


for disease in disease_list:

    disease_patients = (
        df.loc[
            df["disease"] == disease,
            "Study_ID"
        ]
        .dropna()
        .unique()
    )

    patient_info[disease] = (
        patient_info["Study_ID"]
        .isin(disease_patients)
        .astype(int)
    )


# ------------------------------------------------------------
# 5. 各疾患の最初の候補記載日
# ------------------------------------------------------------

for disease in disease_list:

    first_date = (
        df.loc[
            df["disease"] == disease
        ]
        .groupby("Study_ID")["visit_date"]
        .min()
        .rename(
            f"{disease}_first_date"
        )
    )

    patient_info = patient_info.merge(
        first_date,
        on="Study_ID",
        how="left"
    )


# ------------------------------------------------------------
# 6. 各疾患の候補ヒット数
# ------------------------------------------------------------

for disease in disease_list:

    hit_count = (
        df.loc[
            df["disease"] == disease
        ]
        .groupby("Study_ID")
        .size()
        .rename(
            f"{disease}_hit_count"
        )
    )

    patient_info = patient_info.merge(
        hit_count,
        on="Study_ID",
        how="left"
    )

    patient_info[
        f"{disease}_hit_count"
    ] = (
        patient_info[
            f"{disease}_hit_count"
        ]
        .fillna(0)
        .astype(int)
    )


# ------------------------------------------------------------
# 7. 疾患数
# ------------------------------------------------------------

patient_info["disease_count"] = (
    patient_info[disease_list]
    .sum(axis=1)
)


# ------------------------------------------------------------
# 8. 保存
# ------------------------------------------------------------

patient_info.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. 結果確認
# ============================================================

print("=" * 70)
print("Step 3 完了")
print("=" * 70)

print(f"出力ファイル:")
print(OUTPUT_FILE)
print()

print(
    f"最終患者数: {len(patient_info):,}"
)

print()

print("疾患別 暫定1")
print("-" * 40)

for disease in disease_list:

    n = patient_info[disease].sum()

    print(
        f"{disease:20s}: "
        f"{n:3d} / {len(patient_info)}"
    )

print()

print("4疾患すべて0の患者")
print("-" * 40)

none = (
    patient_info["disease_count"] == 0
).sum()

print(f"{none}人")

print()

print("疾患数の分布")
print("-" * 40)

print(
    patient_info[
        "disease_count"
    ]
    .value_counts()
    .sort_index()
)

print()

print("先頭20例")
print("-" * 40)

print(
    patient_info[
        [
            "Study_ID",
            "Patient_ID",
            "ind_date",
            "heart_failure",
            "renal_failure",
            "cirrhosis",
            "cancer",
            "disease_count",
        ]
    ]
    .head(20)
    .to_string(index=False)
)